## PX1 Mini-Cam — карта адресов заводской микроконтроллерной прошивки

В заводском обновлении найден файл `CP/ProteusFirmware.s19`. Эта программа анализирует **только формат Motorola S-record, адреса, контрольные суммы и технические идентификаторы**, не исполняет прошивку, не использует учётные данные и не соединяется с серверами Mini-Cam. Оригинальный `ЛП.imgc` не изменяется. Сырые данные S19 остаются на твоём Google Colab, в GitHub передаётся только код анализа.

Открой в Google Colab на Android → **Среда выполнения → Выполнить все**. Приватный отчёт появится на Google Диске как `PX1_SD_S19_MAP_REPORT.txt`.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, struct, shutil
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
src = root / 'Untitled folder' / 'ЛП.imgc'
if not src.is_file():
    src = next((Path(d)/'ЛП.imgc' for d,_,names in os.walk(root) if 'ЛП.imgc' in names), None)
if not src or not src.is_file(): raise FileNotFoundError('Не найден ЛП.imgc')
project=Path('/content/unimgc')
if not project.is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/Shizmob/unimgc.git',str(project)],check=True)
subprocess.run(['make','-C',str(project)],check=True)
part=Path('/content/px1_ro_partition.img')
offset=32901120
size=7912719360
if shutil.disk_usage('/content').free < size + 1024**3:
    raise RuntimeError('Мало свободного места в Colab для копии раздела')
print('Читаю раздел в временный файл Colab (примерно 7.9 ГБ)...')
p=subprocess.Popen([str(project/'unimgc'),str(src)],stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
try:
    toskip=offset
    while toskip:
        data=p.stdout.read(min(4*1024*1024,toskip))
        if not data: raise EOFError('Образ закончился до начала раздела')
        toskip-=len(data)
    left=size
    with part.open('wb') as out:
        while left:
            data=p.stdout.read(min(4*1024*1024,left))
            if not data: raise EOFError('Образ закончился внутри раздела')
            out.write(data)
            left-=len(data)
finally:
    p.stdout.close()
    p.terminate()
    try: p.wait(timeout=10)
    except subprocess.TimeoutExpired: p.kill(); p.wait()
def dbg(command):
    v=subprocess.run(['debugfs','-R',command,str(part)],capture_output=True,timeout=90)
    return v.stdout.decode('utf-8','replace')

# Read-only parsing of a single Motorola S-Record file inside an owner-provided ZIP.
import zipfile, hashlib, collections, re
report=["PX1 Mini-Cam / CP firmware S-record: format and address-map report",
        "Offline metadata analysis. Code not executed; contents not exported."]
def add(k,v):report.append(str(k)+': '+str(v))
archive=Path('/content/px1_factory_firmware.zip')
archive.unlink(missing_ok=True)
dbg('dump /root/factoryupdate.zip '+str(archive))
if not archive.exists():raise FileNotFoundError('Не удалось получить factoryupdate.zip из SD-образа')
types=collections.Counter(); ranges=[]; invalid=0; records=0; headers=[]; startaddrs=[]
sha=hashlib.sha256()
with zipfile.ZipFile(archive,'r') as z:
    name='CP/ProteusFirmware.s19'
    if name not in z.namelist():raise FileNotFoundError('Архив не содержит '+name)
    with z.open(name) as fd:
        for raw in fd:
            sha.update(raw);row=raw.strip()
            if not row:continue
            records+=1
            if records>1500000:raise RuntimeError('Too many firmware records')
            try:
                if len(row)<4 or row[:1]!=b'S':raise ValueError('invalid S-record')
                ty=chr(row[1]); types[ty]+=1
                count=int(row[2:4],16)
                payload=bytes.fromhex(row[4:].decode('ascii'))
                if len(payload)!=count:raise ValueError('bytecount mismatch')
                if ((count+sum(payload))&255)!=255:raise ValueError('checksum failure')
                addrbytes={'0':2,'1':2,'2':3,'3':4,'5':2,'6':3,'7':4,'8':3,'9':2}.get(ty)
                if addrbytes is None:raise ValueError('unknown record type')
                address=int.from_bytes(payload[:addrbytes],'big')
                data=payload[addrbytes:-1]
                if ty in ('1','2','3') and data:
                    ranges.append((address,address+len(data)))
                if ty in ('7','8','9'):startaddrs.append((ty,address))
                if ty=='0':
                    h=''.join(chr(x) if 32<=x<=126 else '.' for x in data[:80])
                    if re.fullmatch(r'[A-Za-z0-9 _./-]{0,80}',h):headers.append(h)
            except Exception:invalid+=1
add('Total nonempty lines',records)
add('S-record types',dict(sorted(types.items())))
add('Invalid record/checksum count',invalid)
add('Full S19 SHA-256',sha.hexdigest())
if ranges:
    ranges.sort()
    segments=[]
    cur_a,cur_b=ranges[0]
    for a,b in ranges[1:]:
        if a<=cur_b:cur_b=max(cur_b,b)
        else:
            segments.append((cur_a,cur_b))
            cur_a,cur_b=a,b
    segments.append((cur_a,cur_b))
    add('Address range global',f'0x{ranges[0][0]:08X} - 0x{max(y for x,y in ranges)-1:08X}')
    add('Contiguous segments count',len(segments))
    add('Sum of disjoint mapped bytes',sum(b-a for a,b in segments))
    for a,b in segments[:25]:
        add('Segment',f'0x{a:08X} - 0x{b-1:08X}, {b-a} bytes')
    if len(segments)>25:add('Further segments omitted',len(segments)-25)
if startaddrs:
    add('Start/entry addresses',', '.join(t+':0x'+format(a,'08X') for t,a in startaddrs[:8]))
if headers:add('Sanitized public-format S0 header text',', '.join(headers[:5]))
add('Platform identification','Not determined automatically from address map')
add('Remote access or vulnerability','Not tested or proven')
out=root/'PX1_SD_S19_MAP_REPORT.txt'
out.write_text('\n'.join(report)+'\n',encoding='utf-8')
archive.unlink(missing_ok=True)
part.unlink(missing_ok=True)
print('\n'.join(report))
print('Готово. Приватный отчёт:',out.name)
